# ODI to Databricks Migration - TRG_DEP_Load
Converted: 2024-07-30 12:00:00
Source File: TRG_DEP_Load.txt

This notebook performs a full load of department data into the target `trg_dep` table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "F", "ETL Job Type (F=Full, I=Incremental)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "100", "ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "999999", "ODI Session Number")

# ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT to_timestamp('1900-01-01', 'yyyy-MM-dd') AS etl_last_extract_time;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT current_timestamp() AS etl_current_extract_time;

In [ ]:
display(spark.sql("""
    SELECT
        '${ETL_JOB_TYPE}' as ETL_JOB_TYPE,
        ${DATASOURCE_NUM_ID} as DATASOURCE_NUM_ID,
        ${ETL_PROC_WID} as ETL_PROC_WID,
        '${ODI_SESS_NO}' as ODI_SESS_NO,
        (SELECT etl_last_extract_time FROM v_etl_last_extract_time) as ETL_LAST_EXTRACT_TIME,
        (SELECT etl_current_extract_time FROM v_etl_current_extract_time) as ETL_CURRENT_EXTRACT_TIME
"""))

# Target Load: TRG_DEP

In [ ]:
%sql
-- SCEN_TASK_NO in {10}
-- SCEN_TASK_NO in {20}
-- SCEN_TASK_NO in {30}
INSERT INTO workspace.hr.trg_dep
(
  department_id,
  department_name,
  manager_id,
  location_id
)
SELECT
  departments.department_id,
  departments.department_name,
  departments.manager_id,
  departments.location_id
FROM
  workspace.hr.departments AS departments;

# Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_records FROM workspace.hr.trg_dep;

# Conversion Notes and Manual Actions Required

1.  **Schema and Table Names:** Oracle schema `HR` has been mapped to `workspace.hr`. Table names `TRG_DEP` and `DEPARTMENTS` have been converted to lowercase `trg_dep` and `departments`.
2.  **Oracle Hints:** The `/*+ APPEND PARALLEL */` hint has been removed as it is not applicable in Databricks Delta Lake and Spark SQL.
3.  **Empty SCEN_TASK_NO:** The empty `SCEN_TASK_NO {10}`, `{20}`, `{30}` have been retained as comments in the main `INSERT` SQL cell.
4.  **Target Table DDL:** This notebook assumes the `workspace.hr.trg_dep` table already exists with a compatible schema. If not, a `CREATE TABLE` statement would be required before the `INSERT`.
    *   Example DDL (assuming Oracle NUMBER(20,0) for IDs, VARCHAR2 for names):
        ```sql
        -- MAGIC %sql
        CREATE TABLE IF NOT EXISTS workspace.hr.trg_dep (
            department_id   BIGINT,
            department_name STRING,
            manager_id      BIGINT,
            location_id     BIGINT
        ) USING DELTA;
        ```

**Manual Actions Required:**
- Ensure the `workspace.hr` schema exists in your Databricks environment.
- Ensure the `workspace.hr.departments` source table exists and is accessible.
- If `workspace.hr.trg_dep` does not exist, add a `CREATE TABLE` statement with appropriate data types before the `INSERT` statement.